##### Window Function Partitioning (WindowSpec)

##### partitionBy()
- is a method used to **organize data into logical groups**.
- If you **omit partitionBy**, Spark is `forced to move all data to a single worker node` to perform the calculation, which can lead to `OutOfMemory errors on large datasets`.

| Feature | DataFrameWriter.partitionBy | Window.partitionBy |
|---------|-----------------------------|--------------------|
| Primary Goal | Physical storage layout on disk. | Logical grouping for calculations. |
| Output | Folders on the file system. | Aggregated or ranked columns in memory. |
| Impact | Speeds up reads (Pruning). |Enables complex analytical queries. |

**Syntax**

      from pyspark.sql.window import Window
      window_spec = Window.partitionBy("department").orderBy("salary")
      df.withColumn("rank", rank().over(window_spec))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

data = [
    ("A", 100),
    ("A", 300),    
    ("A", 200),
    ("A", 300),
    ("A", 400),
    ("A", 300),
    ("A", None),
    ("B", 70),
    ("B", 90),
    ("B", 70),
    ("B", None),
    ("B", 50),
    ("C", 50),
    ("C", 50),
    ("C", 75),
    ("C", 100),
    ("D", 20),
    ("D", 35),
    ("D", 50),
    ("D", 85),
    ("D", 85)
]

df = spark.createDataFrame(data, ["category", "amount"])
df.display()

category,amount
A,100
A,300
A,200
A,300
A,400
A,300
A,null
B,70
B,90
B,70


##### 1) Row number per group

%md
|     Function      |        Description	                            |                    Tie Handling                   |
|-------------------|-------------------------------------------------|---------------------------------------------------|
| **row_number()**	| Assigns a **unique sequential number** to each row.	| **Tied** values receive **different consecutive numbers** (e.g., 1, 2, 3, 4). |
| **rank()**	      | Assigns the **same rank** to **tied** values but leaves **gaps** in the sequence.	| Tied values receive the same rank, and subsequent ranks are **skipped** (e.g., 1, 2, 2, 4). |
| **dense_rank()**	| Assigns the **same rank** to **tied** values but **does not leave gaps**.	| Tied values receive the same rank, and subsequent ranks are **consecutive** (e.g., 1, 2, 2, 3). |

In [0]:
w = Window.partitionBy("category").orderBy("amount")

df2 = df.withColumn("row_num", F.row_number().over(w))
df2.display()

category,amount,row_num
A,null,1
A,100,2
A,200,3
A,300,4
A,300,5
A,300,6
A,400,7
B,null,1
B,50,2
B,70,3


In [0]:
w = Window.partitionBy("category").orderBy(F.desc("amount"))

df3 = df.withColumn("row_num", F.row_number().over(w))
df3.display()

category,amount,row_num
A,400,1
A,300,2
A,300,3
A,300,4
A,200,5
A,100,6
A,null,7
B,90,1
B,70,2
B,70,3


##### 2) Rank per partition

In [0]:
w = Window.partitionBy("category").orderBy(F.desc("amount"))

df4 = df.withColumn("rank", F.rank().over(w))
df4.display()

category,amount,rank
A,400,1
A,300,2
A,300,2
A,300,2
A,200,5
A,100,6
A,null,7
B,90,1
B,70,2
B,70,2


##### 3) Dense rank per partition

In [0]:
w = Window.partitionBy("category").orderBy(F.desc("amount"))

df5 = df.withColumn("dense_rank", F.dense_rank().over(w))
df5.display()

category,amount,dense_rank
A,400,1
A,300,2
A,300,2
A,300,2
A,200,3
A,100,4
A,null,5
B,90,1
B,70,2
B,70,2


##### 4) Maximum value per group without aggregation

In [0]:
w = Window.partitionBy("category")

df6 = df \
  .withColumn("max_amount", F.max("amount").over(w)) \
  .withColumn("min_amount", F.min("amount").over(w))

df6.display()

category,amount,max_amount,min_amount
A,100,400,100
A,300,400,100
A,200,400,100
A,300,400,100
A,400,400,100
A,300,400,100
A,null,400,100
B,70,90,50
B,90,90,50
B,70,90,50


##### 5) Average value per partition

In [0]:
w = Window.partitionBy("category")

df9 = df.withColumn("avg_amount", F.avg("amount").over(w))
df9.display()

category,amount,avg_amount
A,100,266.6666666666667
A,300,266.6666666666667
A,200,266.6666666666667
A,300,266.6666666666667
A,400,266.6666666666667
A,300,266.6666666666667
A,null,266.6666666666667
B,70,70.0
B,90,70.0
B,70,70.0


##### 6) Lag and Lead within partition

**a) Lag (previous row in same group)**

      lag(column, offset=1, default=None)

- `The default is 1 (the immediate previous row)`.
- `For every row, it looks back the specified number of rows within that same bucket`.
- The **first row** of every **partition** will always return the **default value (or null)** because there is **no previous row to look at within that partition**.

In [0]:
w = Window.partitionBy("category").orderBy("amount")

df7 = df \
    .withColumn("prev_amount", F.lag("amount", 1).over(w)) \
    .withColumn("oneprev_amount", F.lag("amount", 2).over(w)) \
    .withColumn("def0_amount", F.lag("amount", 2, 0).over(w)) \
    .withColumn("def1_amount", F.lag("amount", 2, -1).over(w))

display(df7)

category,amount,prev_amount,oneprev_amount,def0_amount,def1_amount
A,null,null,null,0,-1
A,100,null,null,0,-1
A,200,100,null,null,null
A,300,200,100,100,100
A,300,300,200,200,200
A,300,300,300,300,300
A,400,300,300,300,300
B,null,null,null,0,-1
B,50,null,null,0,-1
B,70,50,null,null,null


**b) Lead (next row in same group)**
- `offset: (Optional) Number of rows to look forward`. **Default is 1**.
- `lead("salary", 1)` **looks 1 row ahead**; returns **None** if **no next row exists**.

| Function | Direction | Best Used For... |
|----------|-----------|------------------|
| lead()   | Future (Look forward) | Calculating the time until the next event or the next price change. |
| lag()    | Past (Look backward)  | Calculating the difference from a previous value (e.g., month-over-month growth). |

In [0]:
df8 = df \
    .withColumn("next_amount", F.lead("amount", 1).over(w)) \
    .withColumn("afternext_amount", F.lead("amount", 2).over(w)) \
    .withColumn("def0_amount", F.lead("amount", 2, 0).over(w)) \
    .withColumn("def1_amount", F.lead("amount", 2, -1).over(w))

display(df8)

category,amount,next_amount,afternext_amount,def0_amount,def1_amount
A,null,100,200,200,200
A,100,200,300,300,300
A,200,300,300,300,300
A,300,300,300,300,300
A,300,300,400,400,400
A,300,400,null,0,-1
A,400,null,null,0,-1
B,null,50,70,70,70
B,50,70,70,70,70
B,70,70,90,90,90


##### 7) Running total per partition
- `Cumulative sum within each group (partition), row by row`.

|        Term               |              Meaning                                    |
|---------------------------|---------------------------------------------------------|
| Window.unboundedPreceding | Start from the very first row of the window (partition) |
| Window.currentRow         | up to current row                                       |

**Meaning:** 
- `Take all rows from the beginning of the partition up to this row`.

In [0]:
w = Window.partitionBy("category").orderBy("amount") \
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)

df3 = df.withColumn("running_total", F.sum("amount").over(w))
df3.display()

category,amount,running_total
A,null,null
A,100,100
A,200,300
A,300,600
A,300,900
A,300,1200
A,400,1600
B,null,null
B,50,50
B,70,120


| Row | Amount | Running Total Calculation |
| --- | ------ | ------------------------- |
| 1   | 100    | 100                       |
| 2   | 200    | 100 + 200 = 300           |
| 3   | 300    | 100 + 200 + 300 = 600     |
| 4   | 300    | 100 + 200 + 300 + 300 = 900     |
| 5   | 300    | 100 + 200 + 300 + 300 + 300 = 1200     |
| 6   | 400    | 100 + 200 + 300 + 300 + 300 + 400 = 1600     |

      Partition: A

      Row 1 → [start → current] → [100]
      Row 2 → [start → current] → [100, 200]
      Row 3 → [start → current] → [100, 200, 300]

**Summary Table: Common Window Functions**

| Purpose               | Function          | Needs OrderBy? |
| --------------------- | ----------------- | -------------- |
| Row number            | `row_number()`    | Yes            |
| Rank                  | `rank()`          | Yes            |
| Dense Rank            | `dense_rank()`    | Yes            |
| Running total         | `sum().over()`    | Yes            |
| Max/Min per group     | `max().over()`    | No             |
| Lag/Lead              | `lag()`, `lead()` | Yes            |
| Average per partition | `avg().over()`    | No             |
| Count per partition   | `count().over()`  | No             |